In [3]:
import os, sys, platform, subprocess, torch
print("PyTorch:", torch.__version__)
print("torch.version.cuda:", torch.version.cuda)
print("CUDA available:", torch.cuda.is_available())
print("CUDA device count:", torch.cuda.device_count())
print("CUDA_VISIBLE_DEVICES:", os.environ.get("CUDA_VISIBLE_DEVICES"))
print("Platform:", platform.platform())

# Try to list GPUs via nvidia-smi
try:
    print("\n=== nvidia-smi -L ===")
    print(subprocess.check_output(["nvidia-smi", "-L"], text=True))
except Exception as e:
    print("\nNo nvidia-smi or driver not loaded:", e)


PyTorch: 2.9.0+cpu
torch.version.cuda: None
CUDA available: False
CUDA device count: 0
CUDA_VISIBLE_DEVICES: 0
Platform: Linux-4.18.0-553.50.1.el8_10.x86_64-x86_64-with-glibc2.28

=== nvidia-smi -L ===
GPU 0: NVIDIA L40S (UUID: GPU-be5541bf-be6f-83fe-8c4f-a2050d20decd)



In [6]:
!pip uninstall -y torch torchvision torchaudio


Found existing installation: torch 2.9.0+cpu
Uninstalling torch-2.9.0+cpu:
  Successfully uninstalled torch-2.9.0+cpu
ERROR: Exception:
Traceback (most recent call last):
  File "/software/slurm/spackages/linux-rocky8-x86_64/gcc-12.2.0/anaconda3-2023.09-0-3mhml42fa64byxqyd5fig5tbih625dp2/lib/python3.11/site-packages/pip/_internal/cli/base_command.py", line 180, in exc_logging_wrapper
    status = run_func(*args)
             ^^^^^^^^^^^^^^^
  File "/software/slurm/spackages/linux-rocky8-x86_64/gcc-12.2.0/anaconda3-2023.09-0-3mhml42fa64byxqyd5fig5tbih625dp2/lib/python3.11/site-packages/pip/_internal/commands/uninstall.py", line 110, in run
    uninstall_pathset.commit()
  File "/software/slurm/spackages/linux-rocky8-x86_64/gcc-12.2.0/anaconda3-2023.09-0-3mhml42fa64byxqyd5fig5tbih625dp2/lib/python3.11/site-packages/pip/_internal/req/req_uninstall.py", line 432, in commit
    self._moved_paths.commit()
  File "/software/slurm/spackages/linux-rocky8-x86_64/gcc-12.2.0/anaconda3-2023.09-0-3m

In [7]:
!pip install --index-url https://download.pytorch.org/whl/cu121 torch torchvision torchaudio


Defaulting to user installation because normal site-packages is not writeable
Looking in indexes: https://download.pytorch.org/whl/cu121
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 780.5/780.5 MB 8.7 MB/s eta 0:00:0000:0100:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.7/23.7 MB 25.6 MB/s eta 0:00:0000:0100:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 823.6/823.6 kB 9.2 MB/s eta 0:00:00a 0:00:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 14.1/14.1 MB 39.8 MB/s eta 0:00:0000:0100:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 9.8 MB/s eta 0:00:0000:0100:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 410.6/410.6 MB 15.4 MB/s eta 0:00:0000:0100:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 121.6/121.6 MB 24.4 MB/s eta 0:00:0000:0100:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.5/56.5 MB 40.0 MB/s eta 0:00:0000:0100:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 124.2/124.2 MB 32.2 MB/s eta 0:00:0000:0100:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1

In [1]:
import torch
print(torch.__version__)
print("torch.version.cuda:", torch.version.cuda)
print("CUDA available:", torch.cuda.is_available())
print("CUDA device count:", torch.cuda.device_count())


2.5.1+cu121
torch.version.cuda: 12.1
CUDA available: True
CUDA device count: 1


In [1]:
import os, torch

# Accept either file casing (your files vs. your friend's)
def pick_first_exists(candidates):
    for p in candidates:
        if os.path.exists(p):
            return p
    raise FileNotFoundError(f"None of these files exist: {candidates}")

DATA_DIR = "."

TRAIN_JSON = pick_first_exists([
    os.path.join(DATA_DIR, "Spoken_train-v1.1.json"),
    os.path.join(DATA_DIR, "spoken_train-v1.1.json"),
])

DEV_JSON = pick_first_exists([
    os.path.join(DATA_DIR, "spoken_test-v1.1.json"),
    os.path.join(DATA_DIR, "Spoken_test-v1.1.json"),
])

TEST_WER44_JSON = os.path.join(DATA_DIR, "spoken_test-v1.1_WER44.json")
TEST_WER54_JSON = os.path.join(DATA_DIR, "spoken_test-v1.1_WER54.json")

MODEL_NAME = "bert-base-uncased"   # HF-only per assignment rule
OUTPUT_DIR = "./spoken_squad_bert_base"
os.makedirs(OUTPUT_DIR, exist_ok=True)

MAX_LEN   = 384
DOC_STRIDE = 128
BATCH_SIZE = 8
GRAD_ACCUM_STEPS = 2
EPOCHS = 3
LR = 3e-5
WARMUP_RATIO = 0.1
WEIGHT_DECAY = 0.01
SEED = 42

HAS_CUDA = torch.cuda.is_available()
print("CUDA:", HAS_CUDA)


CUDA: True


In [2]:
import json, random
from datasets import Dataset, DatasetDict
from transformers import AutoTokenizer, default_data_collator, DataCollatorWithPadding, set_seed

set_seed(SEED)

def load_squad_like(path):
    js = json.load(open(path, "r", encoding="utf-8"))
    rows = []
    for art in js["data"]:
        for para in art["paragraphs"]:
            context = para["context"]
            for qa in para["qas"]:
                rows.append({
                    "id": qa.get("id",""),
                    "context": context,
                    "question": qa["question"],
                    "answers": {
                        "text": [a["text"] for a in qa["answers"]],
                        "answer_start": [a["answer_start"] for a in qa["answers"]],
                    }
                })
    return rows

train_examples = load_squad_like(TRAIN_JSON)
dev_examples   = load_squad_like(DEV_JSON)

raw = DatasetDict({
    "train": Dataset.from_list(train_examples),
    "dev":   Dataset.from_list(dev_examples),
})

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, use_fast=True)
PAD_ON_RIGHT = True

def prepare_train_features(ex):
    tokenized = tokenizer(
        ex["question" if PAD_ON_RIGHT else "context"],
        ex["context" if PAD_ON_RIGHT else "question"],
        truncation="only_second" if PAD_ON_RIGHT else "only_first",
        max_length=MAX_LEN,
        stride=DOC_STRIDE,
        return_overflowing_tokens=True,
        return_offsets_mapping=True,
        padding=False,
    )
    sample_mapping = tokenized.pop("overflow_to_sample_mapping")
    offset_mapping = tokenized.pop("offset_mapping")

    start_positions, end_positions = [], []
    for i, offsets in enumerate(offset_mapping):
        input_ids = tokenized["input_ids"][i]
        cls_index = input_ids.index(tokenizer.cls_token_id)
        sample_idx = sample_mapping[i]
        answers = ex["answers"][sample_idx]

        if len(answers["answer_start"]) == 0:
            start_positions.append(cls_index)
            end_positions.append(cls_index)
            continue

        start_char = answers["answer_start"][0]
        end_char   = start_char + len(answers["text"][0])

        seq_ids = tokenized.sequence_ids(i)
        ctx_idx = 1 if PAD_ON_RIGHT else 0
        # find context token span in this window
        tok_start = 0
        while tok_start < len(seq_ids) and seq_ids[tok_start] != ctx_idx:
            tok_start += 1
        tok_end = len(seq_ids) - 1
        while tok_end >= 0 and seq_ids[tok_end] != ctx_idx:
            tok_end -= 1

        if not (offsets[tok_start][0] <= start_char and offsets[tok_end][1] >= end_char):
            start_positions.append(cls_index)
            end_positions.append(cls_index)
        else:
            while tok_start < len(offsets) and offsets[tok_start][0] <= start_char and seq_ids[tok_start] == ctx_idx:
                tok_start += 1
            s_tok = tok_start - 1

            while offsets[tok_end][1] >= end_char and seq_ids[tok_end] == ctx_idx:
                tok_end -= 1
            e_tok = tok_end + 1

            start_positions.append(s_tok)
            end_positions.append(e_tok)

    tokenized["start_positions"] = start_positions
    tokenized["end_positions"] = end_positions
    return tokenized

train_features = raw["train"].map(
    prepare_train_features,
    batched=True,
    remove_columns=raw["train"].column_names,
    desc="Tokenizing train",
)

dev_features = raw["dev"].map(
    prepare_train_features,
    batched=True,
    remove_columns=raw["dev"].column_names,
    desc="Tokenizing dev",
)

# Use dynamic padding (fixes your earlier shape error)
data_collator = DataCollatorWithPadding(tokenizer=tokenizer, pad_to_multiple_of=8)


Tokenizing train:   0%|          | 0/37111 [00:00<?, ? examples/s]

Tokenizing dev:   0%|          | 0/5351 [00:00<?, ? examples/s]

In [3]:
import os, inspect
from transformers import TrainingArguments

os.environ["TOKENIZERS_PARALLELISM"] = "false"

params = {
    "output_dir": OUTPUT_DIR,
    "do_train": True,          # for older versions (replaces evaluation_strategy="no")
    "do_eval": False,
    "learning_rate": LR,
    "per_device_train_batch_size": BATCH_SIZE,
    "per_device_eval_batch_size": BATCH_SIZE,
    "gradient_accumulation_steps": GRAD_ACCUM_STEPS,
    "num_train_epochs": EPOCHS,
    "weight_decay": WEIGHT_DECAY,
    "warmup_ratio": WARMUP_RATIO,
    "seed": SEED,
    "logging_steps": 100,
    "save_steps": 0,
    "dataloader_num_workers": 1,
    "dataloader_pin_memory": HAS_CUDA,
}

sig = inspect.signature(TrainingArguments.__init__)
supported = set(sig.parameters.keys())

# Per-device fallback for very old versions
if "per_device_train_batch_size" not in supported and "per_gpu_train_batch_size" in supported:
    params["per_gpu_train_batch_size"] = params.pop("per_device_train_batch_size")
if "per_device_eval_batch_size" not in supported and "per_gpu_eval_batch_size" in supported:
    params["per_gpu_eval_batch_size"] = params.pop("per_device_eval_batch_size")

args = TrainingArguments(**{k:v for k,v in params.items() if k in supported})

# Precision knobs
import torch
if hasattr(args, "bf16"):
    # bf16 on Ampere+; else fp16 if CUDA
    major, _ = torch.cuda.get_device_capability(0) if torch.cuda.is_available() else (0,0)
    args.bf16 = torch.cuda.is_available() and major >= 8
if hasattr(args, "fp16"):
    args.fp16 = torch.cuda.is_available() and not getattr(args, "bf16", False)

args


TrainingArguments(
_n_gpu=1,
accelerator_config={'split_batches': False, 'dispatch_batches': None, 'even_batches': True, 'use_seedable_sampler': True, 'non_blocking': False, 'gradient_accumulation_kwargs': None, 'use_configured_state': False},
adafactor=False,
adam_beta1=0.9,
adam_beta2=0.999,
adam_epsilon=1e-08,
auto_find_batch_size=False,
average_tokens_across_devices=True,
batch_eval_metrics=False,
bf16=True,
bf16_full_eval=False,
data_seed=None,
dataloader_drop_last=False,
dataloader_num_workers=1,
dataloader_persistent_workers=False,
dataloader_pin_memory=True,
dataloader_prefetch_factor=None,
ddp_backend=None,
ddp_broadcast_buffers=None,
ddp_bucket_cap_mb=None,
ddp_find_unused_parameters=None,
ddp_timeout=1800,
debug=[],
deepspeed=None,
disable_tqdm=False,
do_eval=False,
do_predict=False,
do_train=True,
eval_accumulation_steps=None,
eval_delay=0,
eval_do_concat_batches=True,
eval_on_start=False,
eval_steps=None,
eval_strategy=IntervalStrategy.NO,
eval_use_gather_object=False,
fp1

In [4]:
from transformers import AutoModelForQuestionAnswering, Trainer

model = AutoModelForQuestionAnswering.from_pretrained(MODEL_NAME)

device = "cuda" if torch.cuda.is_available() else "cpu"
_ = model.to(device)
print("Model device:", next(model.parameters()).device)

# Use `tokenizer=` even if it warns (harmless on older versions)
trainer = Trainer(
    model=model,
    args=args,
    train_dataset=train_features,
    eval_dataset=None,
    tokenizer=tokenizer,
    data_collator=data_collator,
)

train_result = trainer.train()
trainer.save_model(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)
train_result


Some weights of BertForQuestionAnswering were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['qa_outputs.bias', 'qa_outputs.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
/local_scratch/slurm.6605225/ipykernel_2085098/960450687.py:10: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(
Detected kernel version 4.18.0, which is below the recommended minimum of 5.5.0; this can cause the process to hang. It is recommended to upgrade the kernel to the minimum version or higher.


Model device: cuda:0


Step,Training Loss
100,5.271400
200,4.273200
300,3.399200
400,2.875100
500,2.530900
600,2.270700
700,2.034800
800,1.965400
900,1.820400
1000,1.761300


TrainOutput(global_step=6996, training_loss=1.2348283824408102, metrics={'train_runtime': 760.4432, 'train_samples_per_second': 147.198, 'train_steps_per_second': 9.2, 'total_flos': 1.4529798103440384e+16, 'train_loss': 1.2348283824408102, 'epoch': 3.0})

In [5]:
import numpy as np, collections, evaluate

squad_metric = evaluate.load("squad")

def prepare_eval_features_no_labels(examples):
    tokenized = tokenizer(
        examples["question" if PAD_ON_RIGHT else "context"],
        examples["context" if PAD_ON_RIGHT else "question"],
        truncation="only_second" if PAD_ON_RIGHT else "only_first",
        max_length=MAX_LEN,
        stride=DOC_STRIDE,
        return_overflowing_tokens=True,
        return_offsets_mapping=True,
        padding=False,
    )
    sample_mapping = tokenized.pop("overflow_to_sample_mapping")
    tokenized["example_id"] = []
    new_offsets = []

    for i, offsets in enumerate(tokenized["offset_mapping"]):
        sample_idx = sample_mapping[i]
        tokenized["example_id"].append(examples["id"][sample_idx])
        # mask non-context offsets so we never select question/CLS/SEP
        seq_ids = tokenized.sequence_ids(i)
        new_offsets.append([
            (o if seq_ids[k] == (1 if PAD_ON_RIGHT else 0) else None)
            for k, o in enumerate(offsets)
        ])
    tokenized["offset_mapping"] = new_offsets
    return tokenized

def postprocess_qa_predictions(examples, features, raw_predictions, n_best_size=20, max_answer_length=30):
    all_start_logits, all_end_logits = raw_predictions

    # examples is a list[dict]; make a dict id -> position
    example_id_to_index = {ex["id"]: i for i, ex in enumerate(examples)}

    # For each feature row, group by its example id
    features_per_example = collections.defaultdict(list)
    for i, f in enumerate(features):
        features_per_example[f["example_id"]].append(i)

    predictions = collections.OrderedDict()
    for example in examples:
        example_id = example["id"]
        context = example["context"]
        feature_indices = features_per_example.get(example_id, [])

        valid_answers = []
        for fi in feature_indices:
            start_logits = all_start_logits[fi]
            end_logits   = all_end_logits[fi]
            offsets      = features[fi]["offset_mapping"]

            start_indexes = np.argsort(start_logits)[-n_best_size:][::-1]
            end_indexes   = np.argsort(end_logits)[-n_best_size:][::-1]
            for s in start_indexes:
                for e in end_indexes:
                    if s >= len(offsets) or e >= len(offsets):
                        continue
                    if offsets[s] is None or offsets[e] is None:
                        continue
                    if e < s or (e - s + 1) > max_answer_length:
                        continue
                    start_char = offsets[s][0]
                    end_char   = offsets[e][1]
                    text = context[start_char:end_char]
                    score = float(start_logits[s] + end_logits[e])
                    valid_answers.append({"text": text, "score": score})

        predictions[example_id] = "" if len(valid_answers) == 0 else max(valid_answers, key=lambda x: x["score"])["text"]
    return predictions


def evaluate_split(examples):
    # Start from a HF Dataset and remember the original columns
    ds = Dataset.from_list(examples)
    original_cols = ds.column_names

    # Build features (drop original columns to avoid length mismatch after overflow)
    feats = ds.map(
        prepare_eval_features_no_labels,
        batched=True,
        remove_columns=original_cols,   # <-- key change
        desc="Tokenizing eval",
    )

    # Keep only the tensors the model expects
    model_cols = ["input_ids", "attention_mask", "token_type_ids", "offset_mapping", "example_id"]
    eval_ds = feats.remove_columns([c for c in feats.column_names if c not in model_cols])

    # Predict
    raw_preds = trainer.predict(eval_ds)

    # Post-process
    preds = postprocess_qa_predictions(examples, feats, raw_preds.predictions)

    # SQuAD metric
    refs = [{"id": ex["id"], "answers": ex["answers"]} for ex in examples]
    metrics = squad_metric.compute(
        predictions=[{"id": k, "prediction_text": v} for k, v in preds.items()],
        references=refs
    )
    return metrics, preds



In [6]:
import json, os

# Dev (clean)
dev_metrics, dev_preds = evaluate_split(dev_examples)
print("DEV metrics:", dev_metrics)
with open(os.path.join(OUTPUT_DIR, "predictions_dev.json"), "w", encoding="utf-8") as f:
    json.dump(dev_preds, f, indent=2)

# Test (clean)
test_examples = load_squad_like(DEV_JSON)
test_metrics, test_preds = evaluate_split(test_examples)
print("TEST (clean) metrics:", test_metrics)
with open(os.path.join(OUTPUT_DIR, "predictions_test_clean.json"), "w", encoding="utf-8") as f:
    json.dump(test_preds, f, indent=2)

# Test (WER44) if available
if os.path.exists(TEST_WER44_JSON):
    test44_examples = load_squad_like(TEST_WER44_JSON)
    test44_metrics, test44_preds = evaluate_split(test44_examples)
    print("TEST (WER44) metrics:", test44_metrics)
    with open(os.path.join(OUTPUT_DIR, "predictions_test_WER44.json"), "w", encoding="utf-8") as f:
        json.dump(test44_preds, f, indent=2)

# Test (WER54) if available
if os.path.exists(TEST_WER54_JSON):
    test54_examples = load_squad_like(TEST_WER54_JSON)
    test54_metrics, test54_preds = evaluate_split(test54_examples)
    print("TEST (WER54) metrics:", test54_metrics)
    with open(os.path.join(OUTPUT_DIR, "predictions_test_WER54.json"), "w", encoding="utf-8") as f:
        json.dump(test54_preds, f, indent=2)


Tokenizing eval:   0%|          | 0/5351 [00:00<?, ? examples/s]

DEV metrics: {'exact_match': 63.80115866193235, 'f1': 74.11301282158452}


Tokenizing eval:   0%|          | 0/5351 [00:00<?, ? examples/s]

TEST (clean) metrics: {'exact_match': 63.80115866193235, 'f1': 74.11301282158452}


Tokenizing eval:   0%|          | 0/5351 [00:00<?, ? examples/s]

TEST (WER44) metrics: {'exact_match': 40.740048589048776, 'f1': 55.74870823878205}


Tokenizing eval:   0%|          | 0/5351 [00:00<?, ? examples/s]

TEST (WER54) metrics: {'exact_match': 28.74229116053074, 'f1': 42.395972657429354}
